# Analyse Data

Analyze for ideal split:
- Count fairytales
- Count tokens per fairytale
    - BiLSTM
    - RoBERTa
- Count annotated spans per fairytale
- Look at percentage Label 1 & 2

In [1]:
import os
import json
import numpy as np
import pandas as pd
import spacy
from spacy.lang.de import German
from math import gcd

In [2]:
INPUT_FILE = "data/grimm_fairytales_annotated/fuzzy_loc_21-8-26_final.json"

In [3]:
with open(INPUT_FILE, "r") as json_file:
    fairytales = json.load(json_file)

In [4]:
def count_words_per_fairytale(input_json):
    words_per_fairytale = {}

    for fairytale in input_json:
        word_count = 0
        text = fairytale["text"]
        words = text.split()
        word_count += len(words)

        words_per_fairytale[fairytale["khm"]] = word_count

    return words_per_fairytale

In [5]:
words_per_fairytale = count_words_per_fairytale(fairytales)
df_fairytale = pd.DataFrame.from_dict(words_per_fairytale, orient="index", columns=["word count"]).sort_index()
df_fairytale.index.name = "khm"
# dictionary mapping khm to text
khm_to_text = {fairytale["khm"]: fairytale["text"] for fairytale in fairytales}
df_fairytale["text"] = df_fairytale.index.map(khm_to_text)

display(df_fairytale)

,word count,text
khm,,
1,1291,"In den alten Zeiten, wo das Wünschen noch geho..."
2,891,Eine Katze hatte Bekanntschaft mit einer Maus ...
3,1771,Vor einem großen Walde lebte ein Holzhacker mi...
4,3418,"Ein Vater hatte zwei Söhne, davon war der älte..."
5,1009,"Es war einmal eine alte Geis, die hatte sieben..."
...,...,...
206,874,"Es war einmal ein Einsiedler, der lebte in ein..."
207,138,"Es hatte einmal ein Fuhrmann seinen Karren, de..."
208,311,Es war in einer großen Stadt ein altes Mütterc...


In [6]:
def sentence_length(df):
    nlp = German()  # contains tokenizer, no other SpaCy Pipeline Components
    nlp.add_pipe("sentencizer")
    for khm, row in df.iterrows():
        text = row["text"]
        sentences = list(nlp(text).sents)

        df.loc[khm, "count sentences"] = int(len(sentences))

        words_per_sentence = [len(sentence.text.split()) for sentence in sentences]
        df.loc[khm, "mean sentence length"] = np.round(np.mean(words_per_sentence))


In [7]:
sentence_length(df_fairytale)
df_fairytale.head()

,word count,text,count sentences,mean sentence length
khm,,,,
1,1291,"In den alten Zeiten, wo das Wünschen noch geho...",52.0,25.0
2,891,Eine Katze hatte Bekanntschaft mit einer Maus ...,49.0,19.0
3,1771,Vor einem großen Walde lebte ein Holzhacker mi...,67.0,26.0
4,3418,"Ein Vater hatte zwei Söhne, davon war der älte...",189.0,18.0
5,1009,"Es war einmal eine alte Geis, die hatte sieben...",50.0,20.0


In [8]:
def count_labels(input_json):
    labels_per_fairytale = {}

    for fairytale in input_json:
        khm = fairytale["khm"]

        total_labels = len(fairytale.get("label", []))
        count_fuzzy_loc = sum(1 for item in fairytale.get("label", [])
                              if item.get("labels", [None])[0] == "FUZZY_LOC")
        count_metaphoric_loc = sum(1 for item in fairytale.get("label", [])
                                   if item.get("labels", [None])[0] == "METAPHORIC_LOC")


        labels_per_fairytale[khm] = (total_labels, count_fuzzy_loc, count_metaphoric_loc)

    return labels_per_fairytale


In [9]:
counted_labels = count_labels(fairytales)

for khm, (total, fuzzy, metaphoric) in counted_labels.items():
    df_fairytale.loc[khm, "total labels"] = total
    df_fairytale.loc[khm, "fuzzy loc count"] = fuzzy
    df_fairytale.loc[khm, "metaphoric loc count"] = metaphoric

num_cols = df_fairytale.select_dtypes(include="number").columns
df_fairytale[num_cols] = df_fairytale[num_cols].astype(int)

df_fairytale.head()

,word count,text,count sentences,mean sentence length,total labels,fuzzy loc count,metaphoric loc count
khm,,,,,,,
1,1291,"In den alten Zeiten, wo das Wünschen noch geho...",52,25,52,48,4
2,891,Eine Katze hatte Bekanntschaft mit einer Maus ...,49,19,18,10,8
3,1771,Vor einem großen Walde lebte ein Holzhacker mi...,67,26,35,30,5
4,3418,"Ein Vater hatte zwei Söhne, davon war der älte...",189,18,80,76,4
5,1009,"Es war einmal eine alte Geis, die hatte sieben...",50,20,38,33,5


In [10]:
def simplify_ratio(a, b):
    if a == 0 and b == 0:
        return "0:0"
    if a == 0:
        return "0:1"
    if b == 0:
        return "1:0"

    divisor = gcd(a, b)
    return f"{a // divisor}:{b // divisor}"

df_fairytale["ratio fuzzy metaphoric"] = df_fairytale.apply(
    lambda row: simplify_ratio(
        row["fuzzy loc count"],
        row["metaphoric loc count"]
    ),
    axis=1
)

df_fairytale.head()

,word count,text,count sentences,mean sentence length,total labels,fuzzy loc count,metaphoric loc count,ratio fuzzy metaphoric
khm,,,,,,,,
1,1291,"In den alten Zeiten, wo das Wünschen noch geho...",52,25,52,48,4,12:1
2,891,Eine Katze hatte Bekanntschaft mit einer Maus ...,49,19,18,10,8,5:4
3,1771,Vor einem großen Walde lebte ein Holzhacker mi...,67,26,35,30,5,6:1
4,3418,"Ein Vater hatte zwei Söhne, davon war der älte...",189,18,80,76,4,19:1
5,1009,"Es war einmal eine alte Geis, die hatte sieben...",50,20,38,33,5,33:5
